In [1]:
%load_ext autoreload
%autoreload 2

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

Note: you may need to restart the kernel to use updated packages.


In [3]:
from src.data import DATA_DIR, load_acs
print (DATA_DIR)

X, y, group = load_acs(2014)
print(X.shape, y.shape, group.shape)

X.head()

/Users/nguyenngochongnhi/Desktop/UTA/Special Topic in AI/Project/fairness-over-time/data/raw
(127039, 10) (127039, 1) (127039, 1)


,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P
0,53.0,1.0,14.0,1.0,5700.0,4.0,1.0,40.0,2.0,1.0
1,68.0,5.0,22.0,1.0,530.0,48.0,0.0,40.0,1.0,1.0
2,64.0,3.0,22.0,1.0,2310.0,48.0,1.0,20.0,2.0,1.0
3,54.0,1.0,21.0,1.0,800.0,48.0,0.0,50.0,1.0,1.0
4,51.0,7.0,18.0,1.0,4700.0,39.0,1.0,40.0,2.0,1.0


## Data dictionary: ACSIncome (Texas, 2014)

Each row is one survey respondent. ACSIncome keeps only people aged 17+, with income over $100, who usually worked at least 1 hour per week, and with a survey weight of at least 1.

### Features (`X`)

| Column | Meaning | Type | Codes |
|---|---|---|---|
| `AGEP` | Age in years | Numeric | 17–92 |
| `COW` | Class of worker (who the person works for) | Nominal | 1 private for-profit · 2 private nonprofit · 3 local gov · 4 state gov · 5 federal gov · 6 self-employed, not incorporated · 7 self-employed, incorporated · 8 unpaid family business |
| `SCHL` | Highest education completed | Ordinal | 1 no schooling … 16 HS diploma · 17 GED · 18–19 some college · 20 associate's · 21 bachelor's · 22 master's · 23 professional degree · 24 doctorate |
| `MAR` | Marital status | Nominal | 1 married · 2 widowed · 3 divorced · 4 separated · 5 never married |
| `OCCP` | Occupation | Nominal | 478 codes in 2014 (codes change in 2018) |
| `POBP` | Place of birth | Nominal | 1–56 US states (48 = Texas) · 60–78 US territories · 100+ other countries |
| `RELP` | Relationship to householder | Nominal | 0 householder · 1 spouse · 2+ child, parent, roommate, etc. |
| `WKHP` | Hours usually worked per week | Numeric | 1–99 (99 = 99 or more) |
| `SEX` | Sex (sensitive attribute) | Nominal | 1 male · 2 female |
| `RAC1P` | Race (sensitive attribute) | Nominal | 1 White · 2 Black · 3 American Indian · 4 Alaska Native · 5 AIAN, tribes specified · 6 Asian · 7 Native Hawaiian/Pacific Islander · 8 Some Other Race · 9 Two or More Races |

### Label (`y`)

| Column | Meaning |
|---|---|
| `PINCP` | True if total personal income (all sources) is over $50,000; False otherwise |

### Sensitive group (`group`)

| Column | Meaning |
|---|---|
| `RAC1P` | Copy of the race column, used for fairness analysis |

*Source: 2014 ACS PUMS Data Dictionary (U.S. Census Bureau).*

In [4]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 127039 entries, 0 to 127038
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   AGEP    127039 non-null  float64
 1   COW     127039 non-null  float64
 2   SCHL    127039 non-null  float64
 3   MAR     127039 non-null  float64
 4   OCCP    127039 non-null  float64
 5   POBP    127039 non-null  float64
 6   RELP    127039 non-null  float64
 7   WKHP    127039 non-null  float64
 8   SEX     127039 non-null  float64
 9   RAC1P   127039 non-null  float64
dtypes: float64(10)
memory usage: 9.7 MB


In [5]:
X.describe()

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P
count,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000
mean,42.690316,2.053275,18.073615,2.415565,4236.300301,86.972977,2.000464,39.755437,1.465715,1.840380
std,14.778244,1.807224,3.757306,1.749398,2580.129698,100.190467,4.052481,13.325580,0.498825,2.063789
min,17.000000,1.000000,1.000000,1.000000,10.000000,1.000000,0.000000,1.000000,1.000000,1.000000
25%,30.000000,1.000000,16.000000,1.000000,2310.000000,47.000000,0.000000,36.000000,1.000000,1.000000
50%,43.000000,1.000000,19.000000,1.000000,4250.000000,48.000000,0.000000,40.000000,1.000000,1.000000
75%,54.000000,3.000000,21.000000,5.000000,5700.000000,48.000000,1.000000,45.000000,2.000000,1.000000
max,92.000000,8.000000,24.000000,5.000000,9830.000000,554.000000,17.000000,99.000000,2.000000,9.000000


In [6]:
print("Duplicate rows:", X.duplicated().sum())

Duplicate rows: 3714


In [7]:
y.value_counts()

PINCP
False    86850
True     40189
Name: count, dtype: int64

In [8]:
group.value_counts()

RAC1P
1        99527
2        12397
8         6294
6         5510
9         2560
3          457
5          192
7           96
4            6
Name: count, dtype: int64

In [9]:
y["PINCP"].groupby(X["SEX"]).mean()

SEX
1.0    0.393473
2.0    0.227875
Name: PINCP, dtype: float64

In [10]:
pd.crosstab(X["RAC1P"], X["SEX"])

SEX,1.0,2.0
RAC1P,,
1.0,53720,45807
2.0,5789,6608
3.0,237,220
4.0,2,4
5.0,110,82
6.0,2936,2574
7.0,52,44
8.0,3645,2649
9.0,1384,1176


In [11]:
import inspect
from folktables import acs
print(inspect.getsource(acs.adult_filter))

def adult_filter(data):
    """Mimic the filters in place for Adult data.

    Adult documentation notes: Extraction was done by Barry Becker from
    the 1994 Census database. A set of reasonably clean records was extracted
    using the following conditions:
    ((AAGE>16) && (AGI>100) && (AFNLWGT>1)&& (HRSWK>0))
    """
    df = data
    df = df[df['AGEP'] > 16]
    df = df[df['PINCP'] > 100]
    df = df[df['WKHP'] > 0]
    df = df[df['PWGTP'] >= 1]
    return df



In [12]:
numeric_cols = ['AGEP']
ordinal_cols = ['SCHL', 'WKHP']
nominal_cols = ['COW', 'MAR', 'OCCP', 'POBP', 'SEX', 'RAC1P', 'RELP']

In [13]:
assert sorted(numeric_cols + ordinal_cols + nominal_cols) == sorted(X.columns)

In [14]:
X[nominal_cols].nunique()

COW        8
MAR        5
OCCP     478
POBP     211
SEX        2
RAC1P      9
RELP      18
dtype: int64

In [54]:
from src.data import split_data
train_idx, val_idx, test_idx = split_data(X, y)

print(len(train_idx), len(val_idx), len(test_idx))

76223 25408 25408


In [50]:
#verify
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(name, y.loc[idx, "PINCP"].mean())

train 0.3163612033113365
val 0.3163176952141058
test 0.31635705289672544


In [ ]:
#verify
print(set(train_idx) & set(val_idx), set(val_idx) & set(test_idx))   # both set()
a, _, _ = split_data(X, y, seed=7)
print(set(a) == set(train_idx))                                       # False

set() set()
False


In [60]:
#saving the split
from src.data import save_split, load_split

split_df = save_split(train_idx, val_idx, test_idx)
print(split_df["split"].value_counts())

# Round-trip check: does what we load back match what we saved?
tr, va, te = load_split()
print(set(tr) == set(train_idx), set(va) == set(val_idx), set(te) == set(test_idx)) 

split_df.head()

split
train    76223
val      25408
test     25408
Name: count, dtype: int64
True True True


,row_id,split
10169,0,val
11171,1,test
12286,2,train
26447,3,train
14337,4,val
